# Étape 7 — Comparaison optionnelle avec la presse lyonnaise

Adaptation de la comparaison de `../../bluesky_topics.ipynb`. Le fichier presse est recherché dans le vrai sous-projet voisin ; les colonnes `km`, `km_label`, `title_clean` sont reconnues. Aucun numéro de thème n’est assimilé entre les deux sources.

La comparaison nécessite un chevauchement des dates et un périmètre compatible. La collecte actuelle est francophone générale et date du 8 novembre, tandis que le corpus presse s’arrête au 1er novembre : aucune comparaison temporelle n’est donc justifiable sur ces données.

In [ ]:
from pathlib import Path
import json
from IPython.display import display

cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le sous-projet ou son dossier notebooks.")

def load_definitions(name, tag):
    path = ROOT / "notebooks" / name
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code" and tag in cell["metadata"].get("tags", []):
            exec(compile("".join(cell["source"]), str(path), "exec"), globals())

load_definitions("00_fonctions_base.ipynb", "database-definitions")
load_definitions("00_fonctions_analyse.ipynb", "analysis-definitions")

PROC = ROOT / "data" / "processed"
TAB = ROOT / "outputs" / "tables"
FIG = ROOT / "outputs" / "figures"
for directory in (PROC, TAB, FIG):
    directory.mkdir(parents=True, exist_ok=True)
print("Sous-projet :", ROOT.name)

In [ ]:
MEDIA_PATH = ROOT.parent / "lyon-news-mining" / "data" / "processed" / "article_topics.parquet"
RUN_COMPARISON = False
SIM_MATCH = 0.45
MIN_ALIGNED_DAYS = 2

path = PROC / "post_topics.csv"
if not path.exists():
    raise FileNotFoundError("Exécuter d'abord les notebooks 04 et 05.")
df = pd.read_csv(path)
df["date"] = pd.to_datetime(df["date"])
TOPIC_COLUMN = "topic" if "topic" in df else "kmeans"
print("Fichier presse présent :", MEDIA_PATH.exists())
print("Dates Bluesky :", df["date"].min(), "→", df["date"].max())

## Comparer les centroides et les series

Comme dans la source, le meme modele encode les deux corpus, les parts sont lissees sur 3 jours et les decalages de -7 a +7 jours sont explores. Les correlations impossibles (moins de deux observations ou variance nulle) restent manquantes. Un decalage descriptif ne prouve pas qu une source publie l information en premier. Les chemins et la jointure article_id sont adaptes aux fichiers existants.

In [ ]:
comparison = pd.DataFrame()
comparison_diagnostics = {"status": "not_run"}
if RUN_COMPARISON:
    if not MEDIA_PATH.exists():
        comparison_diagnostics = {"status": "media_file_missing"}
    else:
        media = load_media_corpus(MEDIA_PATH)
        media_dates = pd.to_datetime(media["date"], utc=True).dt.tz_localize(None).dt.normalize()
        start = max(df["date"].min(), media_dates.min())
        end = min(df["date"].max(), media_dates.max())
        if start > end:
            comparison_diagnostics = {"status": "no_date_overlap"}
        else:
            from sentence_transformers import SentenceTransformer
            encoder = SentenceTransformer(EMB_MODEL)
            comparison, comparison_diagnostics = compare_media_topics(
                df, media, encoder, topic_column=TOPIC_COLUMN,
                min_overlap_days=MIN_ALIGNED_DAYS, similarity_threshold=SIM_MATCH,
            )
else:
    print("Comparaison désactivée : activer RUN_COMPARISON seulement avec des corpus comparables.")
display(pd.Series(comparison_diagnostics, name="Valeur").to_frame())
display(comparison)
if len(comparison.columns):
    comparison.to_csv(TAB / "07_comparaison_medias.csv", index=False)
(PROC / "comparison_diagnostics.json").write_text(
    json.dumps(comparison_diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")